# Reentrenamiento de Umbra: comprobación de resultados
## tl;dr
80 adultos para desarrollo y 30 nuevos para examen. Los cinco regresores nuevos bajan el error frente al histórico, pero ninguno supera la referencia constante en MAE ni alcanza todos los criterios numéricos de Umbra. No corresponde activar puntajes como confiables. Experimento terminado el 8 de septiembre de 2026; producción y tesis sin modificaciones en esta tarea.


## Context & Methods
DistilBERT multilingüe congelado + cinco Ridge nuevos; alternativa TF-IDF + Ridge. Selección en desarrollo mediante validación cruzada anidada 5×5, seed 20260908, alpha 0.1/1/10/100/1000 y menor MSE. Modelos congelados antes del test. Fuentes: [PersonText](https://catalabs.mx/datasets/persontext/), protocolo y adenda en esta carpeta; huellas exactas en training-manifest.json.
### Key Assumptions
Los valores IPIP publicados se multiplican por 100: no equivalen a BFI-2-S, percentiles ni porcentaje de personalidad. Los UID representan participantes; su identidad entre estudios no se pudo verificar. N conserva la orientación nominal del CSV, provisional por discrepancia con el artículo. El test procede de la misma colección y de transcripciones de voz, no de onboarding escrito de Umbra.
Los intervalos pareados usan 2.000 remuestreos de participantes; no agregan personas y no están ajustados por comparaciones múltiples. Una repetición de este notebook sólo verifica cálculos: no reentrena ni vuelve independiente al examen.


## Data
### 1. Cargar resultados agregados
CSV, modelos y predicciones individuales permanecen privados y fuera de Git. Este notebook no muestra textos, identificadores ni puntajes individuales. La carpeta privada es una copia permanente del experimento original; no depende de servicios externos.


In [1]:
from pathlib import Path
import json
import sys
work = Path('/Users/matiasvelez/Developer/umbra/plans/reentrenamiento-persontext-2026-09-08')
private = Path('/Users/matiasvelez/Documents/Umbra-Reentrega-2026-09-08/experimento-ml-persontext-privado')
ml_root = Path('/private/tmp/umbra-entrega-audit.B6Tyg8/repo/ml')
result = json.loads((work / 'results.json').read_text())
audit = result['admission']
print('Desarrollo:', audit['development']['participants_used'], 'adultos /', audit['development']['records_used'], 'textos')
print('Examen:', audit['test']['participants_used'], 'adultos /', audit['test']['records_used'], 'textos únicos')
print('Duplicados del mismo participante eliminados:', audit['test']['same_participant_duplicate_records_removed'])


Desarrollo: 80 adultos / 210 textos
Examen: 30 adultos / 89 textos únicos
Duplicados del mismo participante eliminados: 1


## Results
### 2. Comparar error y criterios sin seleccionar sólo números favorables
MAE es el error absoluto medio en puntos de la escala reexpresada 0–100: menos es mejor. R²>0,20 y Pearson>0,30, junto a n≥30, son los criterios operativos preexistentes; no una validación psicométrica.


In [2]:
labels = {'openness':'Apertura', 'conscientiousness':'Responsabilidad', 'extraversion':'Extraversión', 'agreeableness':'Amabilidad', 'neuroticism':'Neuroticismo*'}
print(f"{'Rasgo':18s} {'MAE viejo':>10s} {'MAE nuevo':>10s} {'MAE simple':>11s} {'MAE media':>10s} {'R² nuevo':>10s} {'Criterios':>10s}")
for dim, label in labels.items():
    values = [result['metrics'][model][dim]['mae_points'] for model in ('historical_distilbert','new_distilbert','new_tfidf','training_mean')]
    metric = result['metrics']['new_distilbert'][dim]
    print(f"{label:18s} {values[0]:10.2f} {values[1]:10.2f} {values[2]:11.2f} {values[3]:10.2f} {metric['r2']:10.3f} {str(metric['numerical_gate_reached']):>10s}")
passed = sum(result['metrics']['new_distilbert'][dim]['numerical_gate_reached'] for dim in labels)
print('Dimensiones que alcanzan todos los criterios:', passed, '/ 5')
assert passed == 0


Rasgo               MAE viejo  MAE nuevo  MAE simple  MAE media   R² nuevo  Criterios
Apertura                22.32      15.87       14.72      13.40     -0.742      False
Responsabilidad         23.02      12.22       12.13      12.13     -0.031      False
Extraversión            24.90      20.78       20.69      20.73     -0.044      False
Amabilidad              27.41      15.60       15.58      15.55     -0.003      False
Neuroticismo*           28.28      21.88       19.50      19.50     -0.448      False
Dimensiones que alcanzan todos los criterios: 0 / 5


### 3. Recalcular por otra vía y verificar modelos guardados
El verificador compara 110 valores con scikit-learn/SciPy, reproduce 15 intervalos pareados con un bucle independiente, comprueba hashes, separación, vocabulario TF-IDF, serialización y corridas MLflow. No ajusta modelos ni cambia criterios.


In [3]:
sys.path.insert(0, str(work))
from verify_evidence import verify
checked = verify(private, ml_root, work)
print('Verificación:', checked['passed'])
print('Métricas recalculadas:', checked['metric_cells_recomputed'], '| diferencia máxima:', checked['metric_max_absolute_discrepancy'])
print('Intervalos recalculados:', checked['paired_ci_cells_recomputed'], '| diferencia máxima:', checked['paired_ci_max_absolute_discrepancy'])
print('Modelos congelados antes del examen:', checked['frozen_before_test'])
print('Originales sin cambios:', checked['inputs_and_originals_unchanged'])
print('Corridas MLflow:', [run['status'] for run in checked['mlflow_runs'].values()])


Verificación: True
Métricas recalculadas: 110 | diferencia máxima: 0.0
Intervalos recalculados: 15 | diferencia máxima: 0.0
Modelos congelados antes del examen: True
Originales sin cambios: True
Corridas MLflow: ['FINISHED', 'FINISHED']


## Takeaways
El nuevo entrenamiento es evidencia de trabajo de ingeniería y evaluación honesta; no acredita precisión suficiente de las cinco estimaciones individuales ni la eficacia de consejos. La arquitectura se mantuvo, sin una búsqueda posterior de modelos para mejorar el examen. La evidencia disponible no justifica quitar el aviso de insuficiencia ni reemplazar producción.
Para la tesis: documentar el resultado y su límite; no afirmar que el objetivo completo de inferencia automática ya se cumplió. Para el usuario: separar puntajes de cuestionario de hipótesis del predictor textual. Más pruebas de software o repetir estos datos no equivalen a nuevas personas reales.
### Estado de ejecución
Las celdas Python se ejecutaron en orden mediante execute_notebook.py, con comprobación estructural básica de notebook 4.5. No se instaló ni utilizó un kernel Jupyter: nbformat, nbclient e ipykernel no están disponibles en este entorno. La ejecución mediante Jupyter propiamente dicho queda sin verificar; instrucciones en COMO_REPETIR.md.
El veredicto completo, intervalos, restricciones académicas y privacidad están en VEREDICTO.md.
